# 2 - Machine Learning - % Gordura

### 2.1 - Configurações Iniciais

In [ ]:
# Testes
import cv2

# drive
from google.colab.patches import cv2_imshow
from google.colab import drive  # importa google drive
# diretórios
path = "/content/drive/MyDrive/Clube de Programacao-2026/Apresentacao/"
path_imagens = f"{path}imagens/"
path_saida_ml = f"{path}machine_learning/"
path_modelo = f"{path_saida_ml}modelo/"
file_modelo_gordura_pkl = f"{path_modelo}modelo_gordura.pkl"

# treinar modelo
from sklearn.neural_network import MLPClassifier
import numpy as np
import joblib
import os

In [ ]:
drive.mount('/content/drive', force_remount=True)  # conecta drive

### 2.2 - Treinar modelo

In [ ]:
# treinar modelo
def treinar_modelo_ml():
    # Conjunto de exemplos utilizado para treinar a rede neural
    # Cada linha representa m pixel no formato HSV
    # Para melhor a acurácia do model é necessário uma maior quantidade de pixel que representem gordura e carne
    X = np.array([[0,0,220],   [0,10,210],   [0,20,230],   [10,15,240],    # gordura
                  [0,180,120], [10,200,100], [5,170,110],  [15,190,130]])  # carne

    # Classes correspondentes aos exemplos de treinamento [1]
    y = np.array([1,1,1,1,   # gordura
                  0,0,0,0])  # carne

    # Cria a rede neural MLP (Multi-Layer Perceptron)
    # Só aumentar os neurônios nesse caso não funciona, pois a amostragem de pixel é a mesma
    modelo = MLPClassifier(
        hidden_layer_sizes=(10,8),  # Define duas camadas ocultas com 10 e 8 neurônios
        activation='relu',          # Função ReLU, ela ativa os neurônios duranteo processo
        solver='adam',              # Algoritmo Adam para otimização, define como a rede neural ajusta seus pesos durante o treinamento quando ela erra
        max_iter=500,               # Número de ajustes de peso durante o treinamento
        random_state=42             # Garante resultados reproduzíveis, a rede começa sempre no mesmo ponto
    )

    modelo.fit(X, y)        # Treinar
    joblib.dump(modelo, file_modelo_gordura_pkl)    # Salvar modelo treinado

    # verifica se o modelo de treinamento foi salvo com sucesso
    if os.path.exists(file_modelo_gordura_pkl):
        print("Modelo treinado e salvo com sucesso.")

# --------------------------------------------------------------------
# treina o modelo com faixas de pixel e salva o modelo treinado
# --------------------------------------------------------------------
treinar_modelo_ml()

### 2.3 - Processar imagens com modelo treinado

In [ ]:
def processar_imagem_ml(imagem, indice):
    imagem = cv2.imread(imagem)

    # Converter para HSV
    hsv = cv2.cvtColor(imagem, cv2.COLOR_BGR2HSV)

    # Detectar fundo branco
    mask_fundo = cv2.inRange(hsv, np.array([0, 0, 200]), np.array([180, 40, 255]))

    # Inverter para obter apenas a peça
    mask_peca = cv2.bitwise_not(mask_fundo)

    # Classificar pixels usando o modelo treinado
    pixels = hsv.reshape((-1, 3))        # Transforma a imagem HSV em uma lista de pixels [H, S, V]
    previsoes = modelo.predict(pixels)   # Usa o modelo treinado para classificar cada pixel como carne ou gordura
    mask_gordura = previsoes.reshape(hsv.shape[0], hsv.shape[1]).astype(np.uint8)    # Reconstrói a imagem com o resultado das classificações
    mask_gordura *= 255             # Converte os valores de 0/1 para 0/255 (preto/branco)

    mask_gordura = cv2.bitwise_and(mask_gordura, mask_peca)   # Mantém apenas as regiões dentro da peça de carne

    kernel = np.ones((5, 5), np.uint8)    # Cria kernel para remoção de ruídos
    mask_gordura = cv2.morphologyEx(mask_gordura, cv2.MORPH_OPEN, kernel)    # Remove pequenos pontos isolados
    mask_gordura = cv2.morphologyEx(mask_gordura, cv2.MORPH_CLOSE, kernel)   # Fecha pequenos buracos dentro das regiões detectadas

    # Cálculo percentual
    pixels_totais = cv2.countNonZero(mask_peca)      # Conta quantos pixels pertencem à peça de carne
    pixels_gordura = cv2.countNonZero(mask_gordura)  # Conta quantos pixels foram classificados como gordura

    # Calcula o percentual de gordura em relação à área da peça
    percentual = (pixels_gordura / pixels_totais) * 100

    print(f"Percentual estimado de gordura: {percentual:.2f}%")
    resultado = imagem.copy()                    # Cria uma cópia da imagem original para exibir o resultado
    resultado[mask_gordura > 0] = [0, 0, 255]    # Destaca em vermelho os pixels classificados como gordura
    fator = 0.5  # reduz o tamnho da imagem exibida

    # exibe as imagens e os resultados após a execução
    print("Imagem Original:")
    cv2_imshow(cv2.resize(imagem,None,fx=fator,fy=fator))
    print("ML - Gordura detectada:")
    cv2_imshow(cv2.resize(mask_gordura,None,fx=fator,fy=fator))
    print("Resultado Final:")
    cv2_imshow(cv2.resize(resultado,None,fx=fator,fy=fator))

    # salva no drive os resultados
    cv2.imwrite(f"{path_saida_ml}{indice:02d}_original.png", imagem)
    cv2.imwrite(f"{path_saida_ml}{indice:02d}_gordura.png", mask_gordura)
    cv2.imwrite(f"{path_saida_ml}{indice:02d}_resultado.png", resultado)

### 2.4 - Testes

In [ ]:
# TESTANDO ------------------------------
modelo = joblib.load(file_modelo_gordura_pkl)

# total de 7 imagens
for i in range(1, 8):
    imagem = f"{path_imagens}{i:02d}.png"

    print("\n========================")
    print(f"Imagem {i:02d}")
    print("========================")

    processar_imagem_ml(imagem, i)